In [1]:
!pip install --upgrade langchain langchain-experimental langchain-openai python-dotenv pyvis

  Using cached langchain_openai-1.1.6-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached langgraph-1.0.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (1.1 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langgraph_prebuilt-1.0.5-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached ht

In [18]:
text = """
Neo4j extracts CDC information from the transaction log. However, by default the transaction log does not contain information directly usable by CDC. For CDC to work, the transaction log need to be enriched with further information. This is applied as an extra configuration option to each database. As soon as CDC is enabled, the database is ready to answer CDC queries from client applications.
CDC has three working modes:
OFF — CDC is disabled (default).
DIFF — Changes are captured as the difference between before and after states of each changed entity (i.e. they only contain removals, updates and additions).
FULL — Changes are recorded as a complete copy of the before and after states of each changed entity (i.e. the contain the full node/relationship, regardless of the extent to which they were altered).
Toggle CDC mode

Create a database with CDC enabled

To create a new database with CDC enabled, use the CREATE DATABASE Cypher command and set the option txLogEnrichment to either FULL or DIFF.
Query
CREATE DATABASE <dbName> IF NOT EXISTS OPTIONS {txLogEnrichment: "FULL"}
Modify a database’s CDC mode

To tweak the CDC mode on an existing database, use the ALTER DATABASE Cypher command and set the option txLogEnrichment to either FULL or DIFF.
Query
ALTER DATABASE <dbName> SET OPTION txLogEnrichment "DIFF"
Modifying CDC mode from DIFF to FULL or viceversa immediately changes the structure of captured changes. Your CDC application must be able to deal with the change of format.
Get a database’s CDC mode

To see what value the CDC mode of a database is, use the SHOW DATABASES Cypher command.
Query
SHOW DATABASES YIELD name, options
Table 1. Result
name	options
"neo4j"
{"txLogEnrichment": "DIFF"}
"system"
{}

"""

In [ ]:
from config import Config
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document


config = Config()

llm = ChatOpenAI(model = config.model, temperature = 0)


graph_transformer = LLMGraphTransformer(llm = llm)

documents = [Document(page_content = text)]
graph_documents = await graph_transformer.aconvert_to_graph_documents(documents)

In [20]:
print(f"Nodes: {graph_documents[0].nodes}")
print(f"Relationships: {graph_documents[0].relationships}")

Nodes: [Node(id='Neo4J', type='Database', properties={}), Node(id='Cdc', type='Concept', properties={}), Node(id='Diff', type='Mode', properties={}), Node(id='Full', type='Mode', properties={}), Node(id='Off', type='Mode', properties={}), Node(id='Create_Database', type='Command', properties={}), Node(id='Alter_Database', type='Command', properties={}), Node(id='Show_Databases', type='Command', properties={})]
Relationships: [Relationship(source=Node(id='Neo4J', type='Database', properties={}), target=Node(id='Cdc', type='Concept', properties={}), type='EXTRACTS', properties={}), Relationship(source=Node(id='Cdc', type='Concept', properties={}), target=Node(id='Diff', type='Mode', properties={}), type='HAS_MODE', properties={}), Relationship(source=Node(id='Cdc', type='Concept', properties={}), target=Node(id='Full', type='Mode', properties={}), type='HAS_MODE', properties={}), Relationship(source=Node(id='Cdc', type='Concept', properties={}), target=Node(id='Off', type='Mode', propert

In [21]:
import os
from pyvis.network import Network

def visualize_graph(graph_documents):
    """
    Visualizes a knowledge graph using PyVis based on the extracted graph documents.

    Args:
        graph_documents (list): A list of GraphDocument objects with nodes and relationships.

    Returns:
        pyvis.network.Network: The visualized network graph object.
    """
    # Create network
    net = Network(height="1200px", width="100%", directed=True,
                    notebook=False, bgcolor="#222222", font_color="white", filter_menu=True, cdn_resources='remote') 

    nodes = graph_documents[0].nodes
    relationships = graph_documents[0].relationships

    # Build lookup for valid nodes
    node_dict = {node.id: node for node in nodes}
    
    # Filter out invalid edges and collect valid node IDs
    valid_edges = []
    valid_node_ids = set()
    for rel in relationships:
        if rel.source.id in node_dict and rel.target.id in node_dict:
            valid_edges.append(rel)
            valid_node_ids.update([rel.source.id, rel.target.id])

    # Track which nodes are part of any relationship
    connected_node_ids = set()
    for rel in relationships:
        connected_node_ids.add(rel.source.id)
        connected_node_ids.add(rel.target.id)

    # Add valid nodes to the graph
    for node_id in valid_node_ids:
        node = node_dict[node_id]
        try:
            net.add_node(node.id, label=node.id, title=node.type, group=node.type)
        except:
            continue  # Skip node if error occurs

    # Add valid edges to the graph
    for rel in valid_edges:
        try:
            net.add_edge(rel.source.id, rel.target.id, label=rel.type.lower())
        except:
            continue  # Skip edge if error occurs

    # Configure graph layout and physics
    net.set_options("""
        {
            "physics": {
                "forceAtlas2Based": {
                    "gravitationalConstant": -100,
                    "centralGravity": 0.01,
                    "springLength": 200,
                    "springConstant": 0.08
                },
                "minVelocity": 0.75,
                "solver": "forceAtlas2Based"
            }
        }
    """)

    output_file = "knowledge_graph.html"
    try:
        net.save_graph(output_file)
        print(f"Graph saved to {os.path.abspath(output_file)}")
        return net
    except Exception as e:
        print(f"Error saving graph: {e}")
        return None

In [22]:
visualize_graph(graph_documents)

Graph saved to /Users/kushalbanda/AI-Engineer/KnowledgeGraph/knowledge_graph.html


<class 'pyvis.network.Network'> |N|=8 |E|=7